In [ ]:

import argparse
import json
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
TARGET = "Exited"
IDENTIFIER_COLUMNS = ["RowNumber", "CustomerId", "Surname"]


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()


def main():
    parser = argparse.ArgumentParser(description="Group 8 Random Forest churn project")
    parser.add_argument("--data", default="Churn_Modelling.csv", help="Path to Churn_Modelling.csv")
    parser.add_argument("--output", default=None, help="Output directory (default: beside this script)")
    # Use parse_known_args() to ignore Colab's default kernel arguments (like -f)
    args, _ = parser.parse_known_args()
    data_path = Path(args.data).expanduser()
    if not data_path.exists():
        # Also try the uploaded Colab location.
        fallback = Path("/content/Churn_Modelling.csv")
        if fallback.exists():
            data_path = fallback
        else:
            raise FileNotFoundError(f"Could not find dataset: {args.data}. Put Churn_Modelling.csv beside the script or pass --data PATH.")
    out = Path(args.output) if args.output else Path(__file__).resolve().parent / "Group8_Project_Outputs" if "__file__" in globals() else Path("./Group8_Project_Outputs")
    out.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid")

    # 1. Data loading and inspection
    df = pd.read_csv(data_path)
    required = {TARGET, "Geography", "Gender"}
    if not required.issubset(df.columns):
        raise ValueError(f"Dataset is missing required columns: {sorted(required - set(df.columns))}")
    rows, cols = df.shape
    with open(out / "dataset_inspection.txt", "w", encoding="utf-8") as f:
        f.write(f"Dataset file: {data_path}\nShape: {df.shape}\n\nColumns:\n{df.columns.tolist()}\n\nData types:\n{df.dtypes.to_string()}\n\nFirst 5 rows:\n{df.head().to_string()}\n\nLast 5 rows:\n{df.tail().to_string()}\n\nInfo:\n")
        df.info(buf=f)
        f.write(f"\n\nMissing values by column:\n{df.isna().sum().to_string()}\n")
        f.write(f"\nExact duplicate rows: {int(df.duplicated().sum())}\n")

    missing_before = int(df.isna().sum().sum())
    duplicates_before = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()
    if not set(df[TARGET].dropna().unique()).issubset({0, 1}):
        raise ValueError("Target 'Exited' must contain binary labels 0 and 1.")
    df = df.dropna(subset=[TARGET]).copy()
    df[TARGET] = df[TARGET].astype(int)

    # 2. Target balance / EDA
    target_counts = df[TARGET].value_counts().sort_index()
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x=TARGET, color="steelblue")
    plt.xticks([0, 1], ["Stayed (0)", "Exited (1)"])
    plt.title("Customer Churn Class Distribution")
    plt.xlabel("Customer outcome")
    plt.ylabel("Number of customers")
    savefig(out / "01_target_distribution.png")

    numeric_all = df.select_dtypes(include=np.number).columns.tolist()
    analysis_numeric = [c for c in numeric_all if c not in IDENTIFIER_COLUMNS and c != TARGET]
    for col in [c for c in ["CreditScore", "Age", "Balance", "EstimatedSalary", "Tenure"] if c in df.columns]:
        plt.figure(figsize=(6, 4))
        sns.histplot(df[col], kde=True, bins=30, color="steelblue")
        plt.title(f"Distribution of {col}")
        savefig(out / f"hist_{col}.png")
    if "Age" in df.columns:
        plt.figure(figsize=(6, 4))
        sns.boxplot(data=df, x="Exited", y="Age", color="steelblue")
        plt.title("Age Distribution by Churn Outcome")
        savefig(out / "02_age_boxplot_by_churn.png")
    if "Geography" in df.columns:
        plt.figure(figsize=(7, 4))
        sns.countplot(data=df, x="Geography", hue=TARGET, palette="deep")
        plt.title("Churn Outcome by Geography")
        plt.ylabel("Number of customers")
        savefig(out / "03_churn_by_geography.png")
    if "Gender" in df.columns:
        plt.figure(figsize=(6, 4))
        sns.countplot(data=df, x="Gender", hue=TARGET, palette="deep")
        plt.title("Churn Outcome by Gender")
        savefig(out / "04_churn_by_gender.png")

    # Statistical summary and outlier analysis on relevant numeric predictors.
    stats_rows = []
    outlier_rows = []
    for col in analysis_numeric:
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        q1, q3 = s.quantile(.25), s.quantile(.75)
        iqr = q3 - q1
        outlier_count = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
        stats_rows.append({
            "Feature": col, "Min": s.min(), "Max": s.max(), "Range": s.max()-s.min(),
            "Q1": q1, "Q3": q3, "IQR": iqr, "Mean": s.mean(),
            "Mean_Absolute_Deviation": (s - s.mean()).abs().mean(),
            "Variance": s.var(ddof=1), "Standard_Deviation": s.std(ddof=1),
            "IQR_Outlier_Count": outlier_count
        })
        outlier_rows.append({"Feature": col, "IQR_Outlier_Count": outlier_count,
                             "IQR_Outlier_Percent": 100*outlier_count/len(s) if len(s) else 0})
    stats_df = pd.DataFrame(stats_rows)
    stats_df.to_csv(out / "statistical_summary.csv", index=False)
    pd.DataFrame(outlier_rows).to_csv(out / "outlier_summary.csv", index=False)
    corr_cols = analysis_numeric + [TARGET]
    corr = df[corr_cols].corr(method="pearson")
    corr.to_csv(out / "pearson_correlation.csv")
    plt.figure(figsize=(11, 8))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
    plt.title("Pearson Correlation Heatmap (Numeric Variables)")
    savefig(out / "05_correlation_heatmap.png")
    box_cols = [c for c in ["CreditScore", "Age", "Balance", "EstimatedSalary", "Tenure"] if c in df.columns]
    if box_cols:
        plt.figure(figsize=(10, 5))
        long = df[box_cols].melt(var_name="Feature", value_name="Value")
        sns.boxplot(data=long, x="Feature", y="Value", color="steelblue")
        plt.xticks(rotation=25, ha="right")
        plt.title("Boxplots for Numerical Predictors")
        savefig(out / "06_numeric_boxplots.png")

    # 3. Feature selection: identifiers and surname are excluded because they are IDs/labels,
    # not meaningful generalizable predictors. Target is separated to avoid leakage.
    feature_cols = [c for c in df.columns if c != TARGET and c not in IDENTIFIER_COLUMNS]
    X = df[feature_cols].copy()
    y = df[TARGET].copy()
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ], remainder="drop")
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1,
                                          bootstrap=True, class_weight="balanced_subsample"))
    ])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

    # 4. Tune a small, reproducible hyperparameter grid using stratified CV.
    param_grid = {
        "model__n_estimators": [150, 250],
        "model__max_depth": [None, 12],
        "model__min_samples_leaf": [1, 3]
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(pipeline, param_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True, verbose=0)
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]
    metrics = {
        "rows": int(rows), "columns": int(cols), "rows_after_dedup_target_clean": int(len(df)),
        "missing_values_before": missing_before, "duplicate_rows_before": duplicates_before,
        "stayed_count": int((y == 0).sum()), "exited_count": int((y == 1).sum()),
        "churn_rate_percent": float(100 * (y == 1).mean()),
        "train_rows": int(len(X_train)), "test_rows": int(len(X_test)),
        "best_parameters": search.best_params_, "best_cv_f1": float(search.best_score_),
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, y_prob)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "feature_columns": feature_cols,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "classification_report": classification_report(y_test, y_pred, target_names=["Stayed", "Exited"], output_dict=True, zero_division=0)
    }
    with open(out / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    with open(out / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(classification_report(y_test, y_pred, target_names=["Stayed (0)", "Exited (1)"], zero_division=0))
        f.write("\nBest parameters: " + str(search.best_params_))
        f.write(f"\nBest CV F1: {search.best_score_:.4f}\n")

    # 5. Evaluation visualizations
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Stayed", "Exited"], cmap="Blues", ax=ax, colorbar=False)
    ax.set_title("Confusion Matrix")
    savefig(out / "07_confusion_matrix.png")
    fig, ax = plt.subplots(figsize=(6, 5))
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax, name="Random Forest")
    ax.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
    ax.set_title("ROC Curve")
    ax.legend()
    savefig(out / "08_roc_curve.png")

    # Learning curve is an appropriate performance curve for a forest (not epoch-based).
    train_sizes, train_scores, val_scores = learning_curve(
        best_model, X_train, y_train, cv=cv, scoring="f1",
        train_sizes=np.linspace(.25, 1.0, 4), n_jobs=-1, shuffle=True, random_state=RANDOM_STATE)
    lc = pd.DataFrame({"Training examples": train_sizes,
                       "Training F1 mean": train_scores.mean(axis=1),
                       "Training F1 std": train_scores.std(axis=1),
                       "Validation F1 mean": val_scores.mean(axis=1),
                       "Validation F1 std": val_scores.std(axis=1)})
    lc.to_csv(out / "learning_curve_values.csv", index=False)
    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training F1")
    plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Cross-validation F1")
    plt.fill_between(train_sizes, train_scores.mean(axis=1)-train_scores.std(axis=1), train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=.15)
    plt.fill_between(train_sizes, val_scores.mean(axis=1)-val_scores.std(axis=1), val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=.15)
    plt.xlabel("Number of training examples")
    plt.ylabel("F1 score")
    plt.title("Random Forest Learning Curve")
    plt.legend()
    savefig(out / "09_learning_curve.png")

    # Feature importances after preprocessing; names map one-hot encoded categories too.
    fitted_pre = best_model.named_steps["preprocessor"]
    fitted_rf = best_model.named_steps["model"]
    try:
        transformed_names = fitted_pre.get_feature_names_out()
        importance = pd.DataFrame({"Feature": transformed_names, "Importance": fitted_rf.feature_importances_}).sort_values("Importance", ascending=False)
    except Exception:
        importance = pd.DataFrame({"Feature": [f"Feature_{i}" for i in range(len(fitted_rf.feature_importances_))], "Importance": fitted_rf.feature_importances_}).sort_values("Importance", ascending=False)
    importance.to_csv(out / "feature_importance.csv", index=False)
    plt.figure(figsize=(9, 6))
    top_imp = importance.head(15).sort_values("Importance")
    plt.barh(top_imp["Feature"], top_imp["Importance"])
    plt.xlabel("Impurity-based importance")
    plt.title("Top 15 Random Forest Feature Importances")
    savefig(out / "10_feature_importance.png")

    # Save test-set predictions with original test indices for review.
    pred_out = X_test.copy()
    pred_out["Actual_Exited"] = y_test
    pred_out["Predicted_Exited"] = y_pred
    pred_out["Probability_Exited"] = y_prob
    pred_out.to_csv(out / "test_predictions.csv", index=False)
    joblib.dump(best_model, out / "random_forest_churn_model.joblib")

    print("\n=== GROUP 8 RANDOM FOREST PROJECT COMPLETE ===")
    print(f"Dataset: {data_path}; original shape={rows} x {cols}")
    print(f"Missing values before cleaning: {missing_before}; duplicate rows: {duplicates_before}")
    print(f"Class counts: stayed={(y==0).sum()}, exited={(y==1).sum()}, churn rate={(y==1).mean():.2%}")
    print("Best parameters:", search.best_params_)
    print(f"Accuracy={metrics['accuracy']:.4f}; Precision={metrics['precision']:.4f}; Recall={metrics['recall']:.4f}; F1={metrics['f1_score']:.4f}; ROC-AUC={metrics['roc_auc']:.4f}")
    print("Outputs saved to:", out)
    print("Confusion matrix [ [TN, FP], [FN, TP] ]:", metrics["confusion_matrix"])

if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=FutureWarning)
    main()